In [1]:
# Import Packages
import sys
sys.path.append('/project/IZZY/MolRepres/benchmarking/Methods/')   
from torch_geometric.datasets import QM9
from rdkit import Chem
from rdkit.Geometry import Point3D
from rdkit.Chem import rdMolTransforms
from torch_geometric.data import Data

from torch_geometric.data import Batch
import torch.nn.functional as F
import numpy as np
import math
from rdkit.Chem import rdchem
from rdkit.Geometry import Point3D
from rdkit.Chem import rdMolTransforms, rdMolDescriptors
from typing import Optional
import os
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score 
import torch
from scipy.optimize import differential_evolution
from rdkit import Chem, RDConfig
from rdkit.Chem import ChemicalFeatures
from rdkit.Chem import AllChem
import os 
import sys
sys.path.append('/project/IZZY/molecular-representation/benchmarking/Methods/') 
from equiformer import EquiformerQM9
from rdkit import RDConfig
from rdkit.Chem import ChemicalFeatures

In [16]:
import torch, torch_geometric
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.0.1+cu118
11.8
True
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition


In [17]:
import torch_scatter, torch_sparse
print(torch_scatter.__version__)
print(torch_sparse.__version__)

2.1.2+pt20cu118
0.6.18+pt20cu118


In [2]:
# Device - Checkpoint - Calling the model 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("/project/IZZY/molecular-representation/checkpoints-all/checkpoints_Equiformernew/best_model.pt", map_location=device)
print(checkpoint.keys())

# Equiformer 
model = EquiformerQM9().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val', 'best_test', 'best_mean_val'])


/home/ismail/dig_envi/lib/python3.9/site-packages/torch/cuda/__init__.py:173: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


<All keys matched successfully>

In [3]:
# Converting Pyg into RDKit mol
# MOL2 helpers
_MOL2_TO_RDKit_BOND = {
    "1": rdchem.BondType.SINGLE,
    "2": rdchem.BondType.DOUBLE,
    "3": rdchem.BondType.TRIPLE,
    "am": rdchem.BondType.SINGLE,   # amide bond
    "ar": rdchem.BondType.AROMATIC,
    "du": rdchem.BondType.SINGLE,    
    "un": rdchem.BondType.SINGLE,    
}


def _parse_mol2_atoms_bonds(mol2_path: str):
    """
    Minimal MOL2 parser for @<TRIPOS>ATOM and @<TRIPOS>BOND blocks.
    Returns:
      atoms: list of dicts with keys: idx(1-based), name, x,y,z, sybyl_type, charge(optional)
      bonds: list of dicts with keys: a1(1-based), a2(1-based), btype(str)
    """
    atoms, bonds = [], []
    in_atoms = False
    in_bonds = False

    with open(mol2_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            if line.startswith("@<TRIPOS>"):
                in_atoms = (line == "@<TRIPOS>ATOM")
                in_bonds = (line == "@<TRIPOS>BOND")
                continue

            if in_atoms:
                # Typical MOL2 ATOM line (space-separated):
                # atom_id name x y z type subst_id subst_name charge
                parts = line.split()
                if len(parts) < 6:
                    continue

                atom_id = int(parts[0])
                name = parts[1]
                x, y, z = map(float, parts[2:5])
                sybyl_type = parts[5]
                charge = float(parts[-1]) if len(parts) >= 9 else None

                atoms.append(
                    dict(idx=atom_id, name=name, x=x, y=y, z=z, sybyl_type=sybyl_type, charge=charge)
                )

            elif in_bonds:
                parts = line.split()
                if len(parts) < 4:
                    continue
                a1 = int(parts[1])
                a2 = int(parts[2])
                btype = parts[3]
                bonds.append(dict(a1=a1, a2=a2, btype=btype))

    if not atoms:
        raise ValueError("No @<TRIPOS>ATOM block found / parsed.")
    return atoms, bonds


def _element_from_sybyl(sybyl_type: str, atom_name: str):
    """
    MOL2 SYBYL types look like: C.3, C.ar, N.am, O.2, H, Cl, Br, etc.
    We'll infer element from SYBYL prefix, falling back to atom_name.
    """
    base = sybyl_type.split(".", 1)[0].strip()
    if base:
        return base
    return atom_name[:2].strip().title()

# Main function
def mol2_to_rdkit_mol(
    mol2_path: str,
    sanitize: bool = True,
    keep_hs: bool = True,
    store_charges: bool = True,
):
    """
    Build an RDKit Mol from a Tripos MOL2 file and attach the 3D conformer from MOL2 coordinates.

    - keep_hs=True keeps explicit H atoms as provided by MOL2 (your example includes H1..H21).
    - store_charges=True writes per-atom 'mol2_charge' property.
    """
    atoms, bonds = _parse_mol2_atoms_bonds(mol2_path)

    # MOL2 atom ids are 1-based; map them to 0-based RDKit indices
    id_to_rd_idx = {}

    rw = Chem.RWMol()

    # Add atoms
    for a in atoms:
        elem = _element_from_sybyl(a["sybyl_type"], a["name"])
        atom = Chem.Atom(elem)

        # store original MOL2 metadata
        atom.SetProp("mol2_name", a["name"])
        atom.SetProp("mol2_type", a["sybyl_type"])
        if store_charges and a["charge"] is not None:
            atom.SetDoubleProp("mol2_charge", float(a["charge"]))

        rd_idx = rw.AddAtom(atom)
        id_to_rd_idx[a["idx"]] = rd_idx

    # Add bonds
    for b in bonds:
        i = id_to_rd_idx[b["a1"]]
        j = id_to_rd_idx[b["a2"]]
        btype = b["btype"].lower()

        rd_bond = _MOL2_TO_RDKit_BOND.get(btype, rdchem.BondType.SINGLE)
        rw.AddBond(i, j, rd_bond)
        if btype == "ar":
            rw.GetBondBetweenAtoms(i, j).SetIsAromatic(True)
            rw.GetAtomWithIdx(i).SetIsAromatic(True)
            rw.GetAtomWithIdx(j).SetIsAromatic(True)

    mol = rw.GetMol()
    # Attach coordinates as conformer
    conf = Chem.Conformer(mol.GetNumAtoms())
    for a in atoms:
        i = id_to_rd_idx[a["idx"]]
        conf.SetAtomPosition(i, Point3D(float(a["x"]), float(a["y"]), float(a["z"])))

    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)
    
    if not keep_hs:
        mol = Chem.RemoveHs(mol, sanitize=sanitize)

    # Sanitize 
    if sanitize:
        Chem.SanitizeMol(mol)
        
    return mol

In [4]:
from rdkit import Chem

def read_sdf_mol(
    sdf_path: str,
    *,
    sanitize: bool = True,
    remove_hs: bool = False,
    largest_fragment: bool = True,
    require_3d: bool = False,
    mol_index: int = 0,
):
    """
    Read one molecule from an SDF file into an RDKit Mol.

    Parameters
    ----------
    sanitize : bool
        Whether to sanitize the molecule.
    remove_hs : bool
        If True, remove explicit hydrogens.
    largest_fragment : bool
        Keep only the largest fragment (recommended for ligands).
    require_3d : bool
        If True, raise error if no conformer present.
    mol_index : int
        Which molecule to read from the SDF (default: first).

    Returns
    -------
    RDKit Mol
    """

    suppl = Chem.SDMolSupplier(
        sdf_path,
        sanitize=sanitize,
        removeHs=False,  # we control H handling ourselves
    )

    if len(suppl) == 0:
        raise ValueError(f"No molecules found in SDF: {sdf_path}")

    mol = suppl[mol_index]

    if mol is None:
        raise ValueError(f"Failed to read molecule at index {mol_index} from {sdf_path}")

    if largest_fragment:
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
        if len(frags) > 1:
            mol = max(frags, key=lambda m: m.GetNumAtoms())

    # Hydrogen handling
    if remove_hs:
        mol = Chem.RemoveHs(mol)
    else:
        mol = Chem.AddHs(mol, addCoords=True)

    # 3D check 
    if require_3d and mol.GetNumConformers() == 0:
        raise ValueError("Molecule has no 3D conformer.")

    return mol

In [5]:
# Get all the torsion angles of one Molecule
def get_all_torsion_angles(
    mol: Chem.Mol,
    *,
    unique: bool = True,
    heavy_only: bool = True,
    rotatable_only: bool = True,
    exclude_rings: bool = True,
    one_per_bond: bool = False,
):
    """
    Extract torsion angles from an RDKit Mol with a 3D conformer.

    Defaults are chosen to be robust for MOL2 ligands:
      - heavy_only=True: ignore torsions involving H atoms
      - rotatable_only=True: keep only rotatable bonds
      - exclude_rings=True: skip bonds in rings
      - unique=True: deduplicate torsions (same 4-tuple up to reversal)
      - one_per_bond=False: if True, pick a single representative (i,j,k,l) per bond
    """

    if mol.GetNumConformers() == 0:
        raise ValueError("Molecule has no 3D coordinates attached")

    conf = mol.GetConformer()
    rot_bonds = None
    if rotatable_only:
        rot_bonds = set(tuple(sorted(x)) for x in rdMolDescriptors.CalcRotatableBonds(mol, strict=True, includeTerminalBonds=False, returnBonds=True))

    torsions = []
    seen = set()
    for bond in mol.GetBonds():
        j = bond.GetBeginAtomIdx()
        k = bond.GetEndAtomIdx()

        # skip ring bonds if desired
        if exclude_rings and bond.IsInRing():
            continue

        # skip non-rotatable if desired
        if rotatable_only:
            if tuple(sorted((j, k))) not in rot_bonds:
                continue

        # neighbors excluding the other end
        nbrs_j = [a.GetIdx() for a in mol.GetAtomWithIdx(j).GetNeighbors() if a.GetIdx() != k]
        nbrs_k = [a.GetIdx() for a in mol.GetAtomWithIdx(k).GetNeighbors() if a.GetIdx() != j]
        if heavy_only:
            nbrs_j = [i for i in nbrs_j if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
            nbrs_k = [l for l in nbrs_k if mol.GetAtomWithIdx(l).GetAtomicNum() > 1]

        if not nbrs_j or not nbrs_k:
            continue

        if one_per_bond:
            pairs = [(min(nbrs_j), min(nbrs_k))]
        else:
            pairs = [(i, l) for i in nbrs_j for l in nbrs_k if i != l]

        for i, l in pairs:
            angle = rdMolTransforms.GetDihedralDeg(conf, int(i), int(j), int(k), int(l))
            atoms = (int(i), int(j), int(k), int(l))

            if unique:
                # torsion (i,j,k,l) == (l,k,j,i)
                key = atoms if atoms < (atoms[3], atoms[2], atoms[1], atoms[0]) else (atoms[3], atoms[2], atoms[1], atoms[0])
                if key in seen:
                    continue
                seen.add(key)

            torsions.append({
                "atoms": atoms,
                "bond": (int(j), int(k)),
                "angle_deg": float(angle),
            })

    return torsions

In [6]:
# Set the torsion angles 
def set_torsion_angle(
    mol: Chem.Mol,
    i: int, j: int, k: int, l: int,
    new_angle: float,
    conf_id: int = 0,
    *,
    copy: bool = True,
    require_bond_jk: bool = True,
    require_path: bool = True,
):
    """
    Set dihedral (i,j,k,l) in degrees on a given conformer.

    Notes for your pipeline:
    - This changes coordinates of many atoms (rotates a fragment around j-k).
    - You must use consistent (i,j,k,l) definitions (ideally one per rotatable bond).
    """
    m = Chem.Mol(mol) if copy else mol  

    n = m.GetNumAtoms()
    for idx in (i, j, k, l):
        if not (0 <= int(idx) < n):
            raise IndexError(f"Atom index {idx} out of range [0, {n-1}]")

    if m.GetNumConformers() == 0:
        raise ValueError("Molecule has no conformer")
    if conf_id < 0 or conf_id >= m.GetNumConformers():
        raise IndexError(f"conf_id={conf_id} out of range (n_confs={m.GetNumConformers()})")

    if require_bond_jk and m.GetBondBetweenAtoms(int(j), int(k)) is None:
        raise ValueError(f"No bond between j={j} and k={k}. Dihedral must be around a bonded pair.")

    if require_path:
        nbrs_j = {a.GetIdx() for a in m.GetAtomWithIdx(int(j)).GetNeighbors()}
        nbrs_k = {a.GetIdx() for a in m.GetAtomWithIdx(int(k)).GetNeighbors()}
        if int(i) not in nbrs_j:
            raise ValueError(f"i={i} is not a neighbor of j={j}. Invalid dihedral definition.")
        if int(l) not in nbrs_k:
            raise ValueError(f"l={l} is not a neighbor of k={k}. Invalid dihedral definition.")

    conf = m.GetConformer(conf_id)

    # wrap angle to a conventional range 
    ang = float(new_angle)
    while ang <= -180.0:
        ang += 360.0
    while ang > 180.0:
        ang -= 360.0
    rdMolTransforms.SetDihedralDeg(conf, int(i), int(j), int(k), int(l), ang)

    return m

In [7]:
# Converting the new Molecule from RDKit to Pyg 
def rdkit_mol_to_pyg_equiformer(
    mol: Chem.Mol,
    *,
    conf_id: int = 0,
    y: Optional[torch.Tensor] = None,
    smiles: Optional[str] = None,
    name: Optional[str] = None,
    idx: Optional[int] = None,
):
    """
    Convert an RDKit Mol with a conformer into a PyG Data that matches the
      x:        [N, 11]
      z:        [N]
      pos:      [N, 3]
      edge_index:[2, 2E]
      edge_attr:[2E, 4]  
      y:        [1, 19]  
    """

    if mol is None:
        raise ValueError("mol is None")
    if mol.GetNumConformers() == 0:
        raise ValueError("RDKit mol has no conformer.")
    if conf_id < 0 or conf_id >= mol.GetNumConformers():
        raise IndexError(f"conf_id={conf_id} out of range (n_confs={mol.GetNumConformers()})")

    conf = mol.GetConformer(conf_id)
    n = mol.GetNumAtoms()

    # pos 
    pos = torch.tensor(
        [[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z] for i in range(n)],
        dtype=torch.float32
    )

    # z 
    z = torch.tensor([mol.GetAtomWithIdx(i).GetAtomicNum() for i in range(n)], dtype=torch.long)

    # x 
    atom_types = [1, 6, 7, 8, 9]  # H,C,N,O,F
    hyb_map = {
        Chem.rdchem.HybridizationType.SP: 0,
        Chem.rdchem.HybridizationType.SP2: 1,
        Chem.rdchem.HybridizationType.SP3: 2,
    }

    x = torch.zeros((n, 11), dtype=torch.float32)
    for i in range(n):
        a = mol.GetAtomWithIdx(i)
        anum = a.GetAtomicNum()

        # one-hot H/C/N/O/F
        if anum in atom_types:
            x[i, atom_types.index(anum)] = 1.0

        # aromatic
        x[i, 5] = 1.0 if a.GetIsAromatic() else 0.0

        # hybridization 
        hyb = a.GetHybridization()
        if hyb in hyb_map:
            x[i, 6 + hyb_map[hyb]] = 1.0

        # charge & H count
        x[i, 9]  = float(a.GetFormalCharge())
        x[i, 10] = float(a.GetTotalNumHs(includeNeighbors=True))

    # edges: edge_index , edge_attr 
    edge_src = []
    edge_dst = []
    edge_attr = []

    def bond_onehot(b: Chem.Bond):
        bt = b.GetBondType()
        if b.GetIsAromatic() or bt == Chem.rdchem.BondType.AROMATIC:
            return [0.0, 0.0, 0.0, 1.0]
        if bt == Chem.rdchem.BondType.SINGLE:
            return [1.0, 0.0, 0.0, 0.0]
        if bt == Chem.rdchem.BondType.DOUBLE:
            return [0.0, 1.0, 0.0, 0.0]
        if bt == Chem.rdchem.BondType.TRIPLE:
            return [0.0, 0.0, 1.0, 0.0]
        return [1.0, 0.0, 0.0, 0.0]

    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()
        attr = bond_onehot(b)
        
        edge_src += [i, j]
        edge_dst += [j, i]
        edge_attr += [attr, attr]

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32)

    # y 
    if y is None:
        y = torch.zeros((1, 19), dtype=torch.float32)
    else:
        y = torch.as_tensor(y).float()
        if y.ndim == 1:
            y = y.view(1, -1)
        if y.shape[0] != 1:
            raise ValueError(f"y must have shape [1, *], got {tuple(y.shape)}")
            
    # smiles 
    if smiles is None:
        smiles = Chem.MolToSmiles(mol, isomericSmiles=True)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        pos=pos,
        z=z,
        smiles=smiles,
    )
    
    if name is not None:
        data.name = name
    if idx is not None:
        data.idx = torch.tensor([idx], dtype=torch.long)

    return data

In [8]:
def prepare_mol_for_pharmacophore(mol: Chem.Mol) -> Chem.Mol:

    m = Chem.Mol(mol)
    m.UpdatePropertyCache(strict=False)

    Chem.FastFindRings(m)
    try:
        Chem.SanitizeMol(
            m,
            sanitizeOps=(
                Chem.SanitizeFlags.SANITIZE_SYMMRINGS |
                Chem.SanitizeFlags.SANITIZE_SETAROMATICITY |
                Chem.SanitizeFlags.SANITIZE_SETCONJUGATION |
                Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION
            )
        )
    except Exception:
        pass

    return m

In [9]:
# # Pharmacophore feature factory (build once)
# feature_definitions = os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef")
# pharma_factory = ChemicalFeatures.BuildFeatureFactory(feature_definitions)

# def pharmaco_features(mol):
#     if mol is None:
#         return []

#     mol_feat = prepare_mol_for_pharmacophore(mol)

#     features = []
#     for feat in pharma_factory.GetFeaturesForMol(mol_feat):
#         features.append({
#             "atom_ids": list(feat.GetAtomIds())
#         })
#     return features


# def pharmacophore_pool(atom_embeddings, pharmacophore_features):
#     feat_embs = []

#     for feat in pharmacophore_features:
#         ids = feat["atom_ids"]
#         if len(ids) == 0:
#             continue

#         feat_emb = atom_embeddings[ids].mean(dim=0)
#         feat_embs.append(feat_emb)

#     if len(feat_embs) == 0:
#         return torch.zeros(atom_embeddings.size(-1), device=atom_embeddings.device)

#     feat_embs = torch.stack(feat_embs, dim=0)   
#     return feat_embs.mean(dim=0)                

In [10]:
# def encode_pharmacophore(model, data, mol, device):
#     with torch.inference_mode():
#         batch = Batch.from_data_list([data]).to(device)

#         # node-level embeddings
#         atom_emb, mask = model.encode(batch)  
#         atom_emb = atom_emb[0]                    

#         pharma_feats = pharmaco_features(mol)
#         mol_emb = pharmacophore_pool(atom_emb, pharma_feats)  

#     return mol_emb.unsqueeze(0)   

In [11]:
# x, out, mask = model.encode(data, return_debug=True)
# print(mask)
# print(mask.shape)

In [12]:
# Import data 
mol2_path = "/project/IZZY/molecular-representation/dataset/DUD-E/urok/crystal_ligand.mol2"

if not os.path.exists(mol2_path):
    raise FileNotFoundError(f"Could not find MOL2 file at: {mol2_path}")

#  MOL2 to RDKit mol 
mol = mol2_to_rdkit_mol(
    mol2_path,
    sanitize=True,
    keep_hs=False,        
    store_charges=True
)

print("RDKit atoms:", mol.GetNumAtoms(), "conformers:", mol.GetNumConformers())

# SMILES string from RDKit 
smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
print("SMILES:", smiles)

# 2) RDKit to PyG Data 
data = rdkit_mol_to_pyg_equiformer(
    mol,
    conf_id=0,
    y=torch.zeros((1, 19), dtype=torch.float32),  
    smiles=smiles,
    name="DUD-E_urok_crystal_ligand",
    idx=0
)
print(data)
print("num atoms (PyG):", data.pos.size(0))
print("edge_index:", tuple(data.edge_index.shape), "edge_attr:", tuple(data.edge_attr.shape))
print("x:", tuple(data.x.shape), "z:", tuple(data.z.shape), "y:", tuple(data.y.shape))

RDKit atoms: 24 conformers: 1
SMILES: COc1ccc2ccc(C(=N)N)cc2c1C1CNN(S(C)(=O)=O)C1
Data(x=[24, 11], edge_index=[2, 52], edge_attr=[52, 4], y=[1, 19], pos=[24, 3], z=[24], smiles='COc1ccc2ccc(C(=N)N)cc2c1C1CNN(S(C)(=O)=O)C1', name='DUD-E_urok_crystal_ligand', idx=[1])
num atoms (PyG): 24
edge_index: (2, 52) edge_attr: (52, 4)
x: (24, 11) z: (24,) y: (1, 19)


In [13]:
def encode_pharmacophore(model, data, mol, device):
    data.pharmacophore_features = model.pharmaco_features(mol)
    batch = Batch.from_data_list([data]).to(device)
    z = model.encode(batch)
    return z

In [15]:
# Pharmacophore-based reference embedding
device = torch.device("cpu")
model = model.to(device)

zt = encode_pharmacophore(model, data, mol, device)
print(zt)

tensor([[ 1.4714e-43,  2.8981e-02,  2.0034e-01, -3.1085e-01,  1.9492e-42,
          1.2803e+00,  2.1019e-43, -9.9542e-01, -1.8598e+00, -8.9664e-01,
          2.1128e+00,  2.2379e-42,  2.6583e-42,  5.8367e-41, -1.8217e-44,
         -1.6788e-42, -2.8968e-02,  2.7472e+00, -2.2000e-43,  1.9646e-42,
         -3.4934e-42, -3.7275e-42, -2.2463e-42, -6.2768e-01,  2.3967e+00,
          9.6587e-01, -1.6824e+00, -2.6979e+00, -2.7816e-42, -1.1979e-01,
          1.2780e-42, -1.5236e-01,  2.2841e-42,  4.0217e-43,  1.1911e-43,
          8.5657e-01, -3.0244e+00, -2.2467e-11,  5.6052e-43,  7.2533e-01,
          1.9604e-42, -3.3407e-42,  1.5891e-42,  1.8321e-01, -2.4300e-01,
         -2.2112e-42,  7.0544e-01,  1.7684e-42,  2.8026e-45,  2.0585e-42,
         -1.7712e-42, -3.6948e-02,  3.3045e-03, -3.6847e-01,  3.9853e-42,
         -1.6788e-42, -3.0857e-42,  3.9797e-43,  1.3564e-03, -1.6587e-01,
          4.7328e-02, -3.2917e-42,  1.5610e-42,  3.5821e-02,  1.6559e+00,
         -3.1481e+00, -4.7266e-03, -4.

In [16]:
# Precompute torsions ONCE
TORSIONS = get_all_torsion_angles(mol, rotatable_only=False)
BOUNDS = [(0.0, 360.0)] * len(TORSIONS)

def objective_theta(theta_vec, mol_current, torsions_current, idx_current, z_ref):
    theta_vec = np.mod(theta_vec, 360.0)

    mol_tmp = Chem.Mol(mol_current)
    for t, theta in zip(torsions_current, theta_vec):
        i, j, k, l = t["atoms"]
        mol_tmp = set_torsion_angle(mol_tmp, i, j, k, l, float(theta), conf_id=0)

    data = rdkit_mol_to_pyg_equiformer(
        mol_tmp,
        conf_id=0,
        y=torch.zeros((1, 19), dtype=torch.float32),
        smiles=Chem.MolToSmiles(mol_tmp, isomericSmiles=True),
        name=mol_tmp.GetProp("_Name") if mol_tmp.HasProp("_Name") else "mol",
        idx=idx_current,
    )
    data.pharmacophore_features = model.pharmaco_features(mol)

    z1 = encode_pharmacophore(model, data, mol_tmp, device)
    cs = F.cosine_similarity(z_ref.to(device), z1, dim=1)

    return float(-cs.item())
# result = differential_evolution(
#     objective_theta,
#     bounds=BOUNDS,
#     strategy="best1bin",
#     maxiter=80,          # increase if you can afford it
#     popsize=10,          # higher = better exploration, more calls
#     tol=1e-3,
#     polish=True,         # local refinement at the end
#     updating="deferred", # faster
#     workers=1           # parallel (uses multiprocessing)
# )

# best_theta_set = np.mod(result.x, 360.0)
# best_cosine = -result.fun

# print("Best cosine:", best_cosine)
# print("Best theta_set:", best_theta_set)

In [17]:
# from scipy.optimize import curve_fit

# def cosine_sim(mol, theta_set):
#     torsions = get_all_torsion_angles(mol, rotatable_only=False)
#     for t, theta in zip(torsions, theta_set):
#         i, j, k, l = t["atoms"]
#         mol_new = set_torsion_angle(mol, i, j, k, l, theta, conf_id=0)
        
#         # RDKit to PyG (Format that Equiformer accept)
#         data = rdkit_mol_to_pyg_equiformer(
#             mol_new,
#             conf_id=0,
#             y=torch.zeros((1, 19), dtype=torch.float32),
#             smiles=Chem.MolToSmiles(mol_new, isomericSmiles=True),
#             name=mol.GetProp("_Name"),
#             idx=idx,
#         )

#         z1 = encode_pharmacophore(model, data, mol_new, device)
#         cs = F.cosine_similarity(zt.to(device), z1, dim=1)
def cosine_sim(mol, theta_set, idx_current=0):
    torsions = get_all_torsion_angles(mol, rotatable_only=False)

    # start from a copy and apply all torsions cumulatively
    mol_new = Chem.Mol(mol)

    for t, theta in zip(torsions, theta_set):
        i, j, k, l = t["atoms"]
        mol_new = set_torsion_angle(mol_new, i, j, k, l, theta, conf_id=0)

    # RDKit -> PyG
    mol_name = mol_new.GetProp("_Name") if mol_new.HasProp("_Name") else "mol"

    data = rdkit_mol_to_pyg_equiformer(
        mol_new,
        conf_id=0,
        y=torch.zeros((1, 19), dtype=torch.float32),
        smiles=Chem.MolToSmiles(mol_new, isomericSmiles=True),
        name=mol_name,
        idx=idx_current,
    )
    data.pharmacophore_features = model.pharmaco_features(mol)
    z1 = encode_pharmacophore(model, data, mol_new, device)
    cs = F.cosine_similarity(zt.to(device), z1, dim=1)

    return cs.squeeze(0).item()

In [ ]:
# Actives and Decoys
folder_active = "/project/IZZY/molecular-representation/dataset/DUD-E/urok/actives_sdf"
folder_decoy  = "/project/IZZY/molecular-representation/dataset/DUD-E/urok/decoys_sdf"

sdf_files_active = sorted(glob(os.path.join(folder_active, "*.sdf")))
sdf_files_decoy  = sorted(glob(os.path.join(folder_decoy, "*.sdf")))
sdf_files = sdf_files_active + sdf_files_decoy

labels = [1] * len(sdf_files_active) + [0] * len(sdf_files_decoy)

cos_sims = []

for idx, (sdf_path, label) in enumerate(
    tqdm(zip(sdf_files, labels), total=len(sdf_files), desc="Processing")
):
    # Read first molecule from the SDF file
    suppl = Chem.SDMolSupplier(sdf_path, sanitize=True, removeHs=False)
    mol = suppl[0] if len(suppl) > 0 else None
    if mol is None:
        continue

    # Tag
    tag = "active" if label == 1 else "decoy"
    mol.SetProp("_Name", f"{tag}_{idx:04d}")
    mol.SetProp("source", "DUD-E_SDF")
    mol.SetProp("sdf_path", sdf_path)

    # Standardize H & require 3D
    mol = Chem.AddHs(mol, addCoords=True)
    if mol.GetNumConformers() == 0:
        continue

    # Precompute torsions once
    TORSIONS = get_all_torsion_angles(mol, rotatable_only=False)

    if TORSIONS:
        BOUNDS = [(0.0, 360.0)] * len(TORSIONS)

        result = differential_evolution(
            lambda theta: objective_theta(theta, mol, TORSIONS, idx, zt),
            bounds=BOUNDS,
            strategy="best1bin",
            maxiter=3,
            popsize=2,
            tol=1e-3,
            polish=True,
            updating="deferred",
            workers=1
        )

        best_theta_set = np.mod(result.x, 360.0)
        best_cosine = -result.fun

    else:
        data = rdkit_mol_to_pyg_equiformer(
            mol,
            conf_id=0,
            y=torch.zeros((1, 19), dtype=torch.float32),
            smiles=Chem.MolToSmiles(mol, isomericSmiles=True),
            name=mol.GetProp("_Name"),
            idx=idx,
        )

        z1 = encode_pharmacophore(model, data, mol, device)
        cs = F.cosine_similarity(zt.to(device), z1, dim=1)
        best_cosine = cs.squeeze(0).detach().cpu().item()

    cos_sims.append(best_cosine)

print("Cos_sim_Crystal_actives_decoys:", cos_sims)

Processing:   0%|          | 6/10003 [01:09<29:49:09, 10.74s/it]

In [ ]:
print("World")